# Paper results: every condition in paper.sh

One row per condition, metrics read from each run's `preds/meta.json` — no re-encoding, runs
on CPU in seconds. The condition list is **hard-coded**: the union of the 2026-08-29 run
(the classic/mse/cosent grid, still valid — nothing that trains them changed since) and the
2026-08-31 run (the infonce/siglip families and both ablations). Default metric everywhere:
**Recall@20**; text panels on the left, image on the right.

Sections: 1 health · 2 easy × V ablation · 3 protocol comparison · 4 win rate vs hard negative · 5 full table

In [ ]:
import json
import os
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from utils.paper_analysis import discover_runs, health_check

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)

NOTE = "paper"
MODELS_ROOT = "models"
K_MAIN = 20                      # the paper's headline cutoff (full table, sorting)
# Text saturates -- at recall@20 most conditions sit above 0.9, and by 2026-09-03 the top
# ten validation cells were within 0.014 of each other at recall@10 as well -- so the text
# panels and the text hparam selection use recall@5 (top-ten spread 0.019). Image is far
# from saturation and stays at @20.
K_TEXT = 5
K_IMAGE = 20
KS = (1, 5, 10, 20, 100)


def k_for(modality):
    return K_TEXT if modality == "text" else K_IMAGE
FIG_DIR = "paper/figs"
os.makedirs(FIG_DIR, exist_ok=True)

## 1. Conditions and health

Columns: modality, style, query_kind, V, easy. `V`/`easy` are None/20 for styles that never
see the measured distance. The merge is on all five keys, so the easy-ablation variants of
one (style, V) stay distinct.

In [ ]:
# 2026-08-29 run -- the pair/margin grid; unchanged by anything since.
CONDITIONS_0829 = [
    ("text", "untrained",        "original", None, 20),
    ("text", "untrained",        "synthetic", None, 20),
    ("text", "untrained",        "rephrased", None, 20),
    ("text", "baseline-triplet", "original", None, 20),
    ("text", "baseline-triplet", "synthetic", None, 20),
    ("text", "baseline-triplet", "rephrased", None, 20),
    ("text", "cosent",           "original", None, 20),
    ("text", "cosent",           "synthetic", None, 20),
    ("text", "cosent",           "rephrased", None, 20),
    ("text", "classic-mse",      "original", 40, 20),
    ("text", "classic-mse",      "synthetic", 40, 20),
    ("text", "classic-mse",      "rephrased", 40, 20),
    ("text", "ours-mse",         "original", 40, 20),
    ("text", "ours-mse",         "synthetic", 40, 20),
    ("text", "ours-mse",         "rephrased", 40, 20),
    ("text", "ours-mse-batched", "original", 40, 20),
    ("text", "ours-mse-batched", "synthetic", 40, 20),
    ("text", "ours-mse-batched", "rephrased", 40, 20),
    ("text", "ours-mse",         "synthetic", 20, 20),
    ("text", "ours-mse",         "synthetic", 60, 20),
    ("text", "ours-mse-batched", "synthetic", 20, 20),
    ("text", "ours-mse-batched", "synthetic", 60, 20),
    ("multimodal", "untrained",        "synthetic", None, 20),
    ("multimodal", "untrained",        "rephrased", None, 20),
    ("multimodal", "baseline-triplet", "synthetic", None, 20),
    ("multimodal", "baseline-triplet", "rephrased", None, 20),
    ("multimodal", "cosent",           "synthetic", None, 20),
    ("multimodal", "cosent",           "rephrased", None, 20),
    ("multimodal", "classic-mse",      "synthetic", 40, 20),
    ("multimodal", "classic-mse",      "rephrased", 40, 20),
    ("multimodal", "ours-mse",         "synthetic", 40, 20),
    ("multimodal", "ours-mse",         "rephrased", 40, 20),
    ("multimodal", "ours-mse-batched", "synthetic", 40, 20),
    ("multimodal", "ours-mse-batched", "rephrased", 40, 20),
    ("multimodal", "ours-mse",         "synthetic", 20, 20),
    ("multimodal", "ours-mse",         "synthetic", 60, 20),
    ("multimodal", "ours-mse-batched", "synthetic", 20, 20),
    ("multimodal", "ours-mse-batched", "synthetic", 60, 20),
]

# 2026-08-31 run -- infonce/siglip families, V ablation, easy ablation.
CONDITIONS_0831 = [
    ("text", "infonce",          "original", None, 20),
    ("text", "infonce",          "synthetic", None, 20),
    ("text", "infonce",          "rephrased", None, 20),
    ("text", "infonce-mined",    "original", None, 20),
    ("text", "infonce-mined",    "synthetic", None, 20),
    ("text", "infonce-mined",    "rephrased", None, 20),
    ("text", "siglip-mined",     "original", None, 20),
    ("text", "siglip-mined",     "synthetic", None, 20),
    ("text", "siglip-mined",     "rephrased", None, 20),
    ("text", "ours-infonce",     "original", 40, 20),
    ("text", "ours-infonce",     "synthetic", 40, 20),
    ("text", "ours-infonce",     "rephrased", 40, 20),
    ("text", "ours-siglip",      "original", 40, 20),
    ("text", "ours-siglip",      "synthetic", 40, 20),
    ("text", "ours-siglip",      "rephrased", 40, 20),
    ("text", "ours-infonce",     "synthetic", 20, 20),
    ("text", "ours-infonce",     "synthetic", 60, 20),
    ("text", "ours-siglip",      "synthetic", 20, 20),
    ("text", "ours-siglip",      "synthetic", 60, 20),
    ("text", "classic-mse",      "synthetic", 40, 30),
    ("text", "classic-mse",      "synthetic", 40, 40),
    ("text", "ours-mse",         "synthetic", 40, 30),
    ("text", "ours-mse",         "synthetic", 40, 40),
    ("text", "ours-mse-batched", "synthetic", 40, 30),
    ("text", "ours-mse-batched", "synthetic", 40, 40),
    ("text", "ours-siglip",      "synthetic", 40, 30),
    ("text", "ours-siglip",      "synthetic", 40, 40),
    ("multimodal", "infonce",          "synthetic", None, 20),
    ("multimodal", "infonce",          "rephrased", None, 20),
    ("multimodal", "infonce-mined",    "synthetic", None, 20),
    ("multimodal", "infonce-mined",    "rephrased", None, 20),
    ("multimodal", "siglip-mined",     "synthetic", None, 20),
    ("multimodal", "siglip-mined",     "rephrased", None, 20),
    ("multimodal", "ours-infonce",     "synthetic", 40, 20),
    ("multimodal", "ours-infonce",     "rephrased", 40, 20),
    ("multimodal", "ours-siglip",      "synthetic", 40, 20),
    ("multimodal", "ours-siglip",      "rephrased", 40, 20),
    ("multimodal", "ours-infonce",     "synthetic", 20, 20),
    ("multimodal", "ours-infonce",     "synthetic", 60, 20),
    ("multimodal", "ours-siglip",      "synthetic", 20, 20),
    ("multimodal", "ours-siglip",      "synthetic", 60, 20),
    ("multimodal", "classic-mse",      "synthetic", 40, 30),
    ("multimodal", "classic-mse",      "synthetic", 40, 40),
    ("multimodal", "ours-mse",         "synthetic", 40, 30),
    ("multimodal", "ours-mse",         "synthetic", 40, 40),
    ("multimodal", "ours-mse-batched", "synthetic", 40, 30),
    ("multimodal", "ours-mse-batched", "synthetic", 40, 40),
    ("multimodal", "ours-siglip",      "synthetic", 40, 30),
    ("multimodal", "ours-siglip",      "synthetic", 40, 40),
]

# ours-infonce-margin run (queued 2026-08-31): mined infonce + distance-scheduled logit
# margins. Main grid V=40, V ablation 10/20/60 on synthetic. Rows show as unhealthy
# ("no preds") until the run completes.
CONDITIONS_MARGIN = [
    ("text",       "ours-infonce-margin", "original",  40, 20),
    ("text",       "ours-infonce-margin", "synthetic", 40, 20),
    ("text",       "ours-infonce-margin", "rephrased", 40, 20),
    ("text",       "ours-infonce-margin", "synthetic", 10, 20),
    ("text",       "ours-infonce-margin", "synthetic", 20, 20),
    ("text",       "ours-infonce-margin", "synthetic", 60, 20),
    ("multimodal", "ours-infonce-margin", "synthetic", 40, 20),
    ("multimodal", "ours-infonce-margin", "rephrased", 40, 20),
    ("multimodal", "ours-infonce-margin", "synthetic", 10, 20),
    ("multimodal", "ours-infonce-margin", "synthetic", 20, 20),
    ("multimodal", "ours-infonce-margin", "synthetic", 60, 20),
]

# mse-mined: ours-mse-batched's binary control, at ours-mse-batched's tuned hparams so the
# two see identical batches and differ in one target cell only.
CONDITIONS_MSE_MINED = [
    ("text",       "mse-mined", "original",  40, 10),
    ("text",       "mse-mined", "synthetic", 40, 10),
    ("text",       "mse-mined", "rephrased", 40, 10),
    ("multimodal", "mse-mined", "synthetic", 80, 10),
    ("multimodal", "mse-mined", "rephrased", 80, 10),
]

conditions = pd.DataFrame(CONDITIONS_0829 + CONDITIONS_0831 + CONDITIONS_MARGIN + CONDITIONS_MSE_MINED,
                          columns=["modality", "style", "query_kind", "V", "easy"])

# The lists above are history: what was run and when. The main grid paper.sh trains *now* is
# read from the script itself, so a retuned row (new V or easy) reaches `results` without
# anyone editing this cell. Duplicates between the two sources collapse to one condition.
from utils.paper_analysis import parse_conditions


def easy_of(extra):
    for token in (extra or "").split(","):
        if token.startswith("easy="):
            return int(token[len("easy="):])
    return 20


current = parse_conditions("paper.sh")
current = current[~current["extra"].fillna("").str.contains("split=val")]
current = pd.DataFrame({"modality": current["modality"], "style": current["style"],
                        "query_kind": current["query_kind"], "V": current["V"],
                        "easy": current["extra"].map(easy_of), "negs": current["negs"]})
# The historical lists predate retrieval-mined negatives; every row there is labeled.
conditions["negs"] = "labeled"
conditions = pd.concat([conditions, current], ignore_index=True)
conditions["V"] = conditions["V"].astype("float64")
conditions = conditions.drop_duplicates().reset_index(drop=True)

runs = discover_runs(MODELS_ROOT, note=NOTE)
runs["easy"] = runs["easy"].fillna(20).astype(int)
matched = conditions.merge(runs, on=["modality", "style", "query_kind", "V", "easy", "negs"], how="left")
matched["has_preds"] = matched["has_preds"].fillna(False).astype(bool)

health = health_check(matched)
print(f"{len(conditions)} conditions | {int(health['healthy'].sum())} usable | "
      f"{int((~health['healthy']).sum())} not")
display(health[~health["healthy"]][["modality", "style", "query_kind", "V", "n_queries", "problem"]])
usable = matched[health["healthy"].to_numpy()].reset_index(drop=True)

In [ ]:
def condition_metrics(row):
    meta = json.load(open(os.path.join(row.run_dir, "preds", "meta.json")))
    out = {"modality": row.modality, "style": row.style, "query_kind": row.query_kind,
           "V": row.V, "easy": row.easy, "negs": row.negs, "n_queries": meta["n_queries"]}
    for k in KS:
        out[f"recall@{k}"] = meta["metrics"][f"recall@{k}"]
    return out

results = pd.DataFrame([condition_metrics(r) for r in usable.itertuples()])
results["query_kind"] = pd.Categorical(results["query_kind"],
                                       ["original", "synthetic", "rephrased"], ordered=True)
# One column that already holds each row's modality-appropriate cutoff, so the ablation
# pivots below stay a single metric column instead of one per modality.
results["recall@k_mod"] = [row[f"recall@{k_for(row['modality'])}"]
                           for _, row in results.iterrows()]
print(f"{len(results)} conditions loaded")

## 2. easy × V ablation

One heatmap per style: the easy-negative distance against the distance normalizer V, on
**synthetic queries**. These were two separate sections, each holding the other knob at its
main-grid default (V=40 / easy=20). The knobs interact, so neither 1-D slice generalises --
for text `ours-mse-batched`, V=80 is the *worst* column at easy=10 and the *best* at easy=40.
A grid shows that; two crosshairs through it cannot.

Read from `preds_val/`, not `preds/`: the grid was swept on the validation split, which is
also the right split to choose V and easy on. The test numbers in later sections are not
touched by this selection.

Only styles with all nine cells are drawn. `classic-mse` and `ours-infonce` were swept at one
point each and are listed under the figure instead.


In [ ]:
# The easy x V grid lives in preds_val/, so it is read from the run dirs rather than reused
# from `results` (which loads test preds). discover_runs is used instead of the hard-coded
# condition table above because this grid was swept after that table was written.
EASY_LEVELS = [10, 20, 40]
V_LEVELS = [20.0, 40.0, 80.0]
V3_STYLE = "infonce-ours-v3"
V3_V_LEVELS = [10.0, 20.0, 40.0, 80.0]


def val_condition(run):
    meta_path = os.path.join(run.run_dir, "preds_val", "meta.json")
    if not os.path.exists(meta_path):
        return None
    meta = json.load(open(meta_path))
    return {"modality": run.modality, "style": run.style, "query_kind": run.query_kind,
            "V": run.V, "easy": run.easy,
            "recall@k_mod": meta["metrics"][f"recall@{k_for(run.modality)}"]}


val_runs = discover_runs(MODELS_ROOT, note=NOTE)
val_runs["easy"] = val_runs["easy"].fillna(20).astype(int)
ablation = pd.DataFrame([row for row in (val_condition(r) for r in val_runs.itertuples())
                         if row is not None])
ablation = ablation[ablation["query_kind"] == "synthetic"]
print(f"{len(ablation)} validation-split synthetic conditions loaded")


In [ ]:
grid = ablation[ablation["easy"].isin(EASY_LEVELS) & ablation["V"].isin(V_LEVELS)]

for modality in ["text", "multimodal"]:
    sub = grid[grid["modality"] == modality]
    # ours-infonce is the deprecated GradedInfoNCELoss (soft-target grading, not the
    # margin loss). It never had a real grid -- 1/9 cells, and easy=10 collides with its
    # own label check in train.py -- and its name reads too close to ours-infonce-margin,
    # so it is dropped here rather than reported as an incomplete style.
    # infonce-ours-v3 has no easy axis (see the next cell), so it is drawn there instead.
    tables = {style: sub[sub["style"] == style]
              .pivot_table(index="easy", columns="V", values="recall@k_mod")
              .reindex(index=EASY_LEVELS, columns=V_LEVELS)
              for style in sorted(sub["style"].unique())
              if style not in ("ours-infonce", V3_STYLE)}
    full = [s for s, t in tables.items() if t.notna().to_numpy().all()]
    sparse = [s for s in tables if s not in full]

    # One colour scale across the row, so cells are comparable between styles and not just
    # within one heatmap.
    lo = min(tables[s].to_numpy().min() for s in full)
    hi = max(tables[s].to_numpy().max() for s in full)
    fig, axes = plt.subplots(1, len(full), figsize=(4.0 * len(full), 3.8), squeeze=False)
    for ax, style in zip(axes[0], full):
        table = tables[style]
        values = table.to_numpy()
        im = ax.imshow(values, cmap="viridis", vmin=lo, vmax=hi, aspect="auto")
        ax.grid(False)  # the seaborn whitegrid theme would draw rules across the cells
        best = values.argmax()
        for i, easy in enumerate(EASY_LEVELS):
            for j, v in enumerate(V_LEVELS):
                value = table.loc[easy, v]
                ax.text(j, i, f"{value:.3f}", ha="center", va="center", fontsize=9,
                        fontweight="bold" if i * len(V_LEVELS) + j == best else "normal",
                        color="white" if value < (lo + hi) / 2 else "black")
        ax.set_xticks(range(len(V_LEVELS)), [str(int(v)) for v in V_LEVELS])
        ax.set_yticks(range(len(EASY_LEVELS)), [str(e) for e in EASY_LEVELS])
        ax.set_xlabel("V")
        ax.set_ylabel("easy-negative distance")
        ax.set_title(style, fontsize=10)
    fig.colorbar(im, ax=axes[0], fraction=0.025, label=f"Recall@{k_for(modality)} (val)")
    # The selection metric differs by modality -- Recall@5 for text, Recall@20 for image,
    # via k_for -- so name it in the title instead of leaving it to the colourbar alone.
    fig.suptitle(f"{'text' if modality == 'text' else 'image'} \u2014 easy x V "
                 "(synthetic, validation split)\n"
                 f"hparams selected on Recall@{k_for(modality)}; bold cell = argmax",
                 fontweight="bold", y=1.10)
    fig.savefig(os.path.join(FIG_DIR, f"easy_v_ablation_{modality}.png"), dpi=150,
                bbox_inches="tight")
    plt.show()
    if sparse:
        print(f"{modality}: incomplete grid, not drawn -- "
              + ", ".join(f"{s} ({int(tables[s].notna().to_numpy().sum())}/9)" for s in sparse))


In [ ]:
# infonce-ours-v3 has no easy axis: random products hold target mass 0 whatever easy is, and
# the label only identifies them. Its sweep is one-dimensional in V, so it is drawn as a line
# per modality with the two references scored on the same validation split: infonce-mined
# (the ungraded control, no V) and ours-infonce-margin at its selected (V=80, easy=10).
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, modality in zip(axes, ["text", "multimodal"]):
    sub = ablation[ablation["modality"] == modality]
    sweep = (sub[sub["style"] == V3_STYLE].set_index("V")["recall@k_mod"]
             .reindex(V3_V_LEVELS))
    ax.plot(V3_V_LEVELS, sweep.to_numpy(), marker="o", label=V3_STYLE)
    if sweep.notna().any():
        best_v = sweep.idxmax()
        ax.annotate(f"V={int(best_v)}: {sweep[best_v]:.3f}", (best_v, sweep[best_v]),
                    textcoords="offset points", xytext=(0, 8), ha="center", fontsize=9)
    for style, sel in (("infonce-mined", sub["V"].isna()),
                       ("ours-infonce-margin", (sub["V"] == 80) & (sub["easy"] == 10))):
        ref = sub[(sub["style"] == style) & sel]
        if not ref.empty:
            ax.axhline(ref["recall@k_mod"].iloc[0], linestyle="--", linewidth=1, label=style)
    ax.set_xscale("log", base=2)
    ax.set_xticks(V3_V_LEVELS, [str(int(v)) for v in V3_V_LEVELS])
    ax.set_xlabel("V")
    ax.set_ylabel(f"Recall@{k_for(modality)} (val)")
    ax.set_title("text" if modality == "text" else "image", fontsize=10)
    ax.legend(fontsize=8)
    missing = [int(v) for v in V3_V_LEVELS if pd.isna(sweep.get(v))]
    if missing:
        ax.text(0.02, 0.02, f"missing V: {missing}", transform=ax.transAxes,
                fontsize=8, color="0.3")
fig.suptitle(f"{V3_STYLE} \u2014 V sweep (synthetic, validation split); "
             f"V selected on Recall@{K_TEXT} text / Recall@{K_IMAGE} image",
             fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "v_sweep_infonce_ours_v3.png"), dpi=150, bbox_inches="tight")
plt.show()


## 3. Protocol comparison: graded vs. ungraded, per family

The claim under test is that **grading** the mined hard negative beats leaving it ungraded,
with everything else held fixed -- same batches, same seen examples, same number of
comparisons, targets the only difference. So each family is a designed pair:

| family | ungraded (mined) | graded |
|---|---|---|
| infonce | `infonce-mined` | `infonce-ours-v3` |
| mse | `mse-mined` | `ours-mse-batched` |
| cosent | `cosent` | `ours-cosent` |
| siglip | `siglip-mined` | `ours-siglip` |

Families are in priority order. `infonce-ours-v3` grades the target (mass e^{-s d/V} on
the hard negative); it replaced `ours-infonce-margin` here on 2026-09-03 (margin's easy x V
sweep stays in section 2). `ours-cosent` is a three-rank ordinal (positive > hard >
random) rather than a distance grading: CoSENT reads label order only, so it has no V or
easy and needs no search. The graded member's hparams come from the validation
easy x V sweep (section 2); the comparison itself is on the **test** split.

**Which rows are plotted** is read from `paper.sh` itself: its non-`split=val` condition rows
are the main grid, at each style's chosen (V, easy). That keeps this figure in step with what
the sweep actually trains and tests, instead of a hand-maintained slice that goes stale when
hparams change. A slot with no result is **drawn as a hatched placeholder with the reason** --
`hparam selection pending` (the slot's graded style has no completed, current validation
sweep behind it -- see `SELECTED` in the cell below), `not in paper.sh` (no condition row)
or `no preds` (row exists, run not finished) -- rather than silently dropped, so a missing
model is visible in the figure. A graded slot is reported only when `SELECTED` names its
(V, easy), and the cell checks that entry against both the sweep argmax and the paper.sh row.

Groups are ordered by mean Recall (Recall@5 text, Recall@20 image) across **synthetic and
rephrased only**, best first; `original` is excluded from the ordering as a non-primary
metric. Missing slots sort last. Section 4 uses the identical rule.


In [ ]:
from utils.paper_analysis import parse_conditions

# Protocol pairs, in priority order: (ungraded mined loss, its graded counterpart).
PROTOCOL = {
    "infonce": ("infonce-mined", "infonce-ours-v3"),
    "mse":     ("mse-mined",     "ours-mse-batched"),
    "cosent":  ("cosent",        "ours-cosent"),
    "siglip":  ("siglip-mined",  "ours-siglip"),
}
# The retrieval-mined (NV-Retriever) baseline: infonce-mined trained on the mine_hard_negs.py
# negatives instead of ours. One model per slot, drawn beside every family as the reference.
NV_MINED = "nv-mined"
ROLE = {NV_MINED: "baseline"}
for family, (ungraded, graded) in PROTOCOL.items():
    ROLE[ungraded] = "ungraded"
    ROLE[graded] = "graded"
QK_ORDER = ["original", "synthetic", "rephrased"]
qk_colour = dict(zip(QK_ORDER, sns.color_palette("colorblind", len(QK_ORDER))))


def easy_of(extra):
    for token in (extra or "").split(","):
        if token.startswith("easy="):
            return int(token[len("easy="):])
    return 20


# The main grid is whatever paper.sh will train and test: its non-val rows, at each style's
# chosen hparams. Read from the script so this figure cannot drift from the sweep.
main_grid = parse_conditions("paper.sh")
main_grid = main_grid[~main_grid["extra"].fillna("").str.contains("split=val")].copy()
# The protocol compares grading against not grading the *labeled* negative; rows trained on
# retrieval-mined negatives (negs=mined) are a separate comparison and are excluded here.
main_grid = main_grid[main_grid["negs"] == "labeled"].copy()
main_grid["easy"] = main_grid["extra"].map(easy_of)


# Hyperparameters selected on the validation sweep (section 2), one entry per graded slot
# whose main-grid row depends on that sweep. paper.sh trains whatever V/easy its row says;
# this table says which of those rows are backed by a completed, current sweep. A slot whose
# entry is None is drawn as the hatched placeholder, not as a result, whatever paper.sh holds.
# When making changes that invalidate model selection (a loss, a dataset, the selection
# metric, or the sweep protocol), reset the affected entries to `None` until a proper hparam
# sweep has completed. Only then select the model: set the entry to the sweep argmax and put
# the same V/easy on the paper.sh row.
#
# 2026-09-05: per-query-kind sweep complete (paper.sh "Per-query-kind hparam search"); every
# slot is the argmax of its own validation grid at recall@5 text / recall@20 image, and
# paper.sh carries the same V/easy on each main-grid row.
SELECTED = {
    ("text",       "infonce-ours-v3",  "original"):  {"V": 10, "easy": 20},
    ("text",       "infonce-ours-v3",  "synthetic"): {"V": 20, "easy": 20},
    ("text",       "infonce-ours-v3",  "rephrased"): {"V": 20, "easy": 20},
    ("multimodal", "infonce-ours-v3",  "synthetic"): {"V": 10, "easy": 20},
    ("multimodal", "infonce-ours-v3",  "rephrased"): {"V": 10, "easy": 20},
    ("text",       "ours-mse-batched", "original"):  {"V": 40, "easy": 10},
    ("text",       "ours-mse-batched", "synthetic"): {"V": 40, "easy": 10},
    ("text",       "ours-mse-batched", "rephrased"): {"V": 40, "easy": 10},
    ("multimodal", "ours-mse-batched", "synthetic"): {"V": 80, "easy": 10},
    ("multimodal", "ours-mse-batched", "rephrased"): {"V": 20, "easy": 10},
    ("text",       "ours-siglip",      "original"):  {"V": 20, "easy": 10},
    ("text",       "ours-siglip",      "synthetic"): {"V": 20, "easy": 10},
    ("text",       "ours-siglip",      "rephrased"): {"V": 20, "easy": 10},
    ("multimodal", "ours-siglip",      "synthetic"): {"V": 20, "easy": 10},
    ("multimodal", "ours-siglip",      "rephrased"): {"V": 40, "easy": 10},
}
# The sweep grid each style is selected over. infonce-ours-v3 has no easy axis, so its
# grid is V alone at the default easy (see section 2).
SWEEP_GRID = {
    "ours-mse-batched": [(easy, V) for easy in EASY_LEVELS for V in V_LEVELS],
    "ours-siglip":      [(easy, V) for easy in EASY_LEVELS for V in V_LEVELS],
    V3_STYLE:           [(20, V) for V in V3_V_LEVELS],
}


def val_argmax(modality, style, kind):
    """(easy, V) with the best validation recall over the style's sweep grid for this query
    kind; raises if any grid cell has no preds_val, since a selection needs the whole grid."""
    cells = {}
    for run in val_runs[(val_runs["modality"] == modality) & (val_runs["style"] == style)
                        & (val_runs["query_kind"] == kind)].itertuples():
        row = val_condition(run)
        if row is not None:
            cells[int(run.easy), float(run.V)] = row["recall@k_mod"]
    grid = SWEEP_GRID[style]
    absent = [cell for cell in grid if cell not in cells]
    if absent:
        raise ValueError(f"{modality} {style} {kind}: sweep incomplete, no preds_val for {absent}")
    return max(grid, key=lambda cell: cells[cell])


def main_row(modality, style, kind):
    """The test-split results row for one protocol slot, or (None, why it is missing)."""
    if style in SWEEP_GRID:
        chosen = SELECTED[modality, style, kind]
        if chosen is None:
            return None, "hparam selection pending"
        best_easy, best_v = val_argmax(modality, style, kind)
        if (chosen["easy"], float(chosen["V"])) != (best_easy, best_v):
            raise ValueError(f"{modality} {style} {kind}: SELECTED says {chosen}, "
                             f"sweep argmax is easy={best_easy}, V={int(best_v)}")
    cond = main_grid[(main_grid["modality"] == modality) & (main_grid["style"] == style)
                     & (main_grid["query_kind"] == kind)]
    if cond.empty:
        return None, "not in paper.sh"
    c = cond.iloc[0]
    if style in SWEEP_GRID and (c["easy"], float(c["V"])) != (chosen["easy"], float(chosen["V"])):
        raise ValueError(f"{modality} {style} {kind}: paper.sh row has easy={c['easy']}, "
                         f"V={int(c['V'])}, SELECTED says {chosen}")
    same_v = results["V"].isna() if pd.isna(c["V"]) else (results["V"] == c["V"])
    hit = results[(results["modality"] == modality) & (results["style"] == style)
                  & (results["query_kind"] == kind) & (results["easy"] == c["easy"]) & same_v
                  & (results["negs"] == "labeled")]
    if hit.empty:
        return None, "no preds"
    return hit.iloc[0], None


def kinds_for(modality):
    """Query kinds the main grid defines for this modality (image has no `original`)."""
    present = set(main_grid[main_grid["modality"] == modality]["query_kind"])
    return [k for k in QK_ORDER if k in present]


rows, missing = [], {}
for modality in ["text", "multimodal"]:
    for ungraded, graded in PROTOCOL.values():
        for style in (ungraded, graded):
            for kind in kinds_for(modality):
                row, why = main_row(modality, style, kind)
                if row is None:
                    missing[modality, style, kind] = why
                else:
                    rows.append(row)
main = pd.DataFrame(rows).reset_index(drop=True)
print(f"{len(main)} protocol slots with results, {len(missing)} missing")


def ordered_groups(modality, styles, value_of):
    """Groups sorted by mean value over synthetic/rephrased, best first; missing last."""
    kinds = kinds_for(modality)
    order_kinds = [k for k in ("synthetic", "rephrased") if k in kinds]

    def key(style):
        vals = [value_of(modality, style, k) for k in order_kinds]
        return -sum(vals) / len(vals) if all(v is not None for v in vals) else float("inf")
    return sorted(styles, key=key), kinds


def draw_missing(ax, i, j, width, kinds, reason, drawn):
    """Hatched placeholder for one empty (group, kind) cell; the reason is written once."""
    x = i + (j - (len(kinds) - 1) / 2) * width
    ax.axvspan(x - width * 0.475, x + width * 0.475, facecolor="0.9", edgecolor="0.6",
               hatch="///", linewidth=0, zorder=0)
    if i not in drawn:
        ax.text(i, 0.5, reason, rotation=90, ha="center", va="center", fontsize=7,
                color="0.3", transform=ax.get_xaxis_transform(), zorder=3)
        drawn.add(i)


def recall_of(modality, style, kind):
    if style == NV_MINED:
        hit = results[(results["modality"] == modality) & (results["style"] == "infonce-mined")
                      & (results["query_kind"] == kind) & (results["negs"] == "mined")]
    else:
        hit = main[(main["modality"] == modality) & (main["style"] == style) & (main["query_kind"] == kind)]
    return None if hit.empty else float(hit[f"recall@{k_for(modality)}"].iloc[0])


for modality in ["text", "multimodal"]:
    for kind in kinds_for(modality):
        if recall_of(modality, NV_MINED, kind) is None:
            missing[modality, NV_MINED, kind] = "no preds (negs=mined row)"


def print_ranking(modality, groups, value_of):
    """Groups best first by mean recall over synthetic + rephrased, with the step to the next
    group in percentage points of recall and as a fraction of the group above."""
    order_kinds = [k for k in ("synthetic", "rephrased") if k in kinds_for(modality)]
    print(f"  {modality}: mean Recall@{k_for(modality)} over {' + '.join(order_kinds)}")
    previous = None
    for g in groups:
        vals = [value_of(modality, g, k) for k in order_kinds]
        if any(v is None for v in vals):
            print(f"    {g:<26}missing"); continue
        score = sum(vals) / len(vals)
        line = f"    {g:<26}{score:.4f}"
        if previous is not None:
            line += f"   {100 * (score - previous):+.2f} pp vs. above ({100 * (score - previous) / previous:+.2f}%)"
        print(line)
        previous = score


for family, pair in PROTOCOL.items():
    print(f"{family}:")
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    for ax, modality in zip(axes, ["text", "multimodal"]):
        groups, kinds = ordered_groups(modality, list(pair) + [NV_MINED], recall_of)
        print_ranking(modality, groups, recall_of)
        width = 0.8 / len(kinds)
        drawn = set()
        for j, k in enumerate(kinds):
            xs = [i + (j - (len(kinds) - 1) / 2) * width for i in range(len(groups))]
            ys = [recall_of(modality, s, k) for s in groups]
            ax.bar(xs, [y if y is not None else 0 for y in ys], width=width * 0.95,
                   color=qk_colour[k], label=k)
            for i, (s, y) in enumerate(zip(groups, ys)):
                if y is None:
                    draw_missing(ax, i, j, width, kinds, missing[modality, s, k], drawn)
        if not any(recall_of(modality, s, k) is not None for s in groups for k in kinds):
            ax.set_ylim(0, 1)
        ax.set_xticks(range(len(groups)))
        ax.set_xticklabels([f"{s}\n({ROLE[s]})" for s in groups], fontsize=8)
        ax.set_title("text" if modality == "text" else "image")
        ax.set_ylabel(f"Recall@{k_for(modality)}")
    axes[0].legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.3), ncol=3, frameon=False)
    fig.suptitle(f"{family}: ungraded vs graded vs nv-mined baseline (test split)", fontweight="bold")
    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, f"protocol_{family}.png"), dpi=150)
    plt.show()

if missing:
    print("missing protocol slots:")
    for (modality, style, kind), why in sorted(missing.items()):
        print(f"  {modality:10} {style:20} {kind:10} {why}")


## 4. Win rate: positive vs. hard negative

Pairwise accuracy: for each (query, positive, hard negative) triple the model **wins** when it
scores the positive above the negative. `sim_pos`/`sim_neg` are already stored per row in every
run's `preds/triplets.jsonl`, so this is a read of existing preds, not a re-encode.

Easy negatives (`negative_example_source == "random"`) are excluded -- they are the random
distractors, not the mined hard negative this experiment is about. Ties count as losses; the
tie count is printed below so it stays visible rather than assumed negligible.

Same protocol pairs, same main-grid rows (read from `paper.sh`), same missing-slot
placeholders and the **same group ordering as section 3**: mean Recall across synthetic and
rephrased, `original` excluded as non-primary. So panels line up bar-for-bar with section 3
and the two metrics can be read across. The y-axis starts at chance (0.5), so bar height is
the margin above a coin flip.


In [ ]:
from utils.distance_labels import EASY_NEGATIVE_SOURCE


def win_rate(run_dir):
    """Fraction of hard-negative pairs whose positive scores above the negative."""
    with open(os.path.join(run_dir, "preds", "triplets.jsonl"), encoding="utf-8") as handle:
        rows = [json.loads(line) for line in handle if line.strip()]
    hard = [r for r in rows if r["negative_example_source"] != EASY_NEGATIVE_SOURCE]
    return {"win_rate": sum(r["sim_pos"] > r["sim_neg"] for r in hard) / len(hard),
            "n_pairs": len(hard),
            "n_ties": sum(r["sim_pos"] == r["sim_neg"] for r in hard)}


wins = pd.DataFrame([
    {"modality": r.modality, "style": r.style, "query_kind": r.query_kind,
     "V": r.V, "easy": r.easy, **win_rate(r.run_dir)}
    for r in usable.itertuples()
])
print(f"{len(wins)} conditions | {wins['n_pairs'].sum():,} hard-negative pairs | "
      f"{wins['n_ties'].sum():,} exact ties (counted as losses)")


In [ ]:
# `main`, `PROTOCOL`, `missing` and the ordering helpers are reused from section 3 so the
# grouping and bar order here cannot drift from that figure.
wins_main = main[["modality", "style", "query_kind", "V", "easy"]].merge(
    wins, on=["modality", "style", "query_kind", "V", "easy"], how="left")
wr = {(r.modality, r.style, r.query_kind): r.win_rate
      for r in wins_main.itertuples() if not pd.isna(r.win_rate)}


def win_of(modality, style, kind):
    return wr[modality, style, kind] if (modality, style, kind) in wr else None


for family, pair in PROTOCOL.items():
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    for ax, modality in zip(axes, ["text", "multimodal"]):
        # Ordered by recall, exactly as section 3, so the panels align.
        groups, kinds = ordered_groups(modality, list(pair), recall_of)
        width = 0.8 / len(kinds)
        drawn = set()
        for j, k in enumerate(kinds):
            xs = [i + (j - (len(kinds) - 1) / 2) * width for i in range(len(groups))]
            ys = [win_of(modality, s, k) for s in groups]
            ax.bar(xs, [y if y is not None else 0 for y in ys], width=width * 0.95,
                   color=qk_colour[k], label=k)
            for i, (s, y) in enumerate(zip(groups, ys)):
                if y is None:
                    why = missing[modality, s, k] if (modality, s, k) in missing else "no triplets"
                    draw_missing(ax, i, j, width, kinds, why, drawn)
        # Floor at chance rather than 0: a win rate below 0.5 would be worse than a coin
        # flip, so bar height reads as the margin above chance.
        ax.set_ylim(bottom=0.5)
        if not any(win_of(modality, s, k) is not None for s in groups for k in kinds):
            ax.set_ylim(0.5, 1)
        ax.set_xticks(range(len(groups)))
        ax.set_xticklabels([f"{s}\n({ROLE[s]})" for s in groups], fontsize=8)
        ax.set_title("text" if modality == "text" else "image")
        ax.set_ylabel("win rate vs hard negative")
    axes[0].legend(fontsize=8)
    fig.suptitle(f"{family}: ungraded vs graded -- win rate", fontweight="bold")
    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, f"winrate_{family}.png"), dpi=150)
    plt.show()

display(wins_main.pivot_table(index=["modality", "style"], columns="query_kind",
                              values="win_rate", observed=True).round(4))


## 5. Best mining baseline vs. ours

Reviewers will read "mined" as retrieval-mined. This section puts the standard mining
pipeline next to our negatives under one loss, then adds our grading on top:

| group | negatives | loss | what it is |
|---|---|---|---|
| `nv-mined` | retrieval-mined, NV-Retriever rule | `infonce-mined` | the best published miner (Moreira et al., CIKM 2025: strong frozen teacher, top-1000 candidates, drop anything above 95% of the positive's score, keep the best survivor) under the strongest ungraded loss |
| `ours-mined` | our labeled hard negative | `infonce-mined` | same loss, our negative source, ungraded |
| `ours-v3` | our labeled hard negative | `infonce-ours-v3` | same negative source, graded target |

`infonce-mined` is the strongest ungraded loss in section 3, so `nv-mined` is the
best-miner-plus-strongest-standard-loss control in one row per query kind. The three
datasets share validation and test rows exactly (mine_hard_negs.py rewrites the train split
only), so every bar is scored on the same corpus and queries.

Reading the gaps: `ours-mined` − `nv-mined` is the value of our negative *source*;
`ours-v3` − `ours-mined` is the value of *grading*; `ours-v3` − `nv-mined` is the paper's
headline against the standard pipeline. The graded group goes through the same `SELECTED`
gate as section 3. Slots are drawn as hatched placeholders until their rows are enabled in
paper.sh (`negs=mined`, currently commented out pending the mined datasets) and have preds.

The table below the figure reads each mined dataset's `mining_report.json`: how many train
queries got an admissible negative, how many the rule dropped, and how often the mined
product coincides with our labeled negative. The drop rate is the method's own behavior on
attribute-conjunction queries and is reported, not tuned.


In [ ]:
# (label, style, negs). negs: labeled = our synthesized negative; mined = the retrieval-mined
# (NV-Retriever) negative, distance unmeasured; mined-graded = the same mined negative after
# label_mined_negs.py measured its distance, so a graded loss can train on it.
MINING_GROUPS = [
    ("ours graded",   "infonce-ours-v3", "labeled"),
    ("ours ungraded", "infonce-mined",   "labeled"),
    ("nv-mined",      "infonce-mined",   "mined"),
    # our grading applied to the retrieval-mined negatives (label_mined_negs.py, 2026-09-09)
    ("ours graded + nv-mined",   "infonce-ours-v3",     "mined-graded"),
    ("margin graded + nv-mined", "ours-infonce-margin", "mined-graded"),
]
# The x-axis orders groups by the mean of their synthetic and rephrased recall, the two
# query kinds both modalities have, so the bars read as a ranking.
RANK_KINDS = ["synthetic", "rephrased"]

# Every non-val row of paper.sh, mined ones included (section 3's main_grid keeps labeled only).
grid_all = parse_conditions("paper.sh")
grid_all = grid_all[~grid_all["extra"].fillna("").str.contains("split=val")].copy()
grid_all["easy"] = grid_all["extra"].map(easy_of)


def mining_row(modality, style, negs, kind):
    """Results row for one group, or (None, why). Labeled slots reuse section 3's lookup and
    its hparam-selection gate; mined and mined-graded slots carry the labeled row's hparams
    on their paper.sh line and match on negs directly."""
    if negs == "labeled":
        return main_row(modality, style, kind)
    cond = grid_all[(grid_all["modality"] == modality) & (grid_all["style"] == style)
                    & (grid_all["query_kind"] == kind) & (grid_all["negs"] == negs)]
    if cond.empty:
        return None, "not in paper.sh"
    c = cond.iloc[0]
    same_v = results["V"].isna() if pd.isna(c["V"]) else (results["V"] == c["V"])
    hit = results[(results["modality"] == modality) & (results["style"] == style)
                  & (results["query_kind"] == kind) & (results["easy"] == c["easy"]) & same_v
                  & (results["negs"] == negs)]
    if hit.empty:
        return None, "no preds"
    return hit.iloc[0], None


mining_values, mining_missing = {}, {}
for modality in ["text", "multimodal"]:
    for label, style, negs in MINING_GROUPS:
        for kind in kinds_for(modality):
            row, why = mining_row(modality, style, negs, kind)
            if row is None:
                mining_missing[modality, label, kind] = why
            else:
                mining_values[modality, label, kind] = float(row[f"recall@{k_for(modality)}"])
print(f"{len(mining_values)} slots with results, {len(mining_missing)} missing")

def rank_score(modality, label):
    return sum(mining_values[modality, label, k] for k in RANK_KINDS) / len(RANK_KINDS)


fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for ax, modality in zip(axes, ["text", "multimodal"]):
    groups = sorted([label for label, _, _ in MINING_GROUPS],
                    key=lambda g: rank_score(modality, g), reverse=True)
    kinds = kinds_for(modality)
    width = 0.8 / len(kinds)
    drawn = set()
    for j, k in enumerate(kinds):
        xs = [i + (j - (len(kinds) - 1) / 2) * width for i in range(len(groups))]
        ys = [mining_values.get((modality, g, k)) for g in groups]
        ax.bar(xs, [y if y is not None else 0 for y in ys], width=width * 0.95,
               color=qk_colour[k], label=k)
        for i, (g, y) in enumerate(zip(groups, ys)):
            if y is None:
                draw_missing(ax, i, j, width, kinds, mining_missing[modality, g, k], drawn)
    if not any((modality, g, k) in mining_values for g in groups for k in kinds):
        ax.set_ylim(0, 1)
    ax.set_xticks(range(len(groups)))
    ax.set_xticklabels(groups, fontsize=8, rotation=15)
    ax.set_title("text" if modality == "text" else "image")
    ax.set_ylabel(f"Recall@{k_for(modality)}")
axes[0].legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=3, frameon=False)
fig.suptitle("Best mining baseline vs. ours (test split)", fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "mining_baseline.png"), dpi=150)
plt.show()

if mining_missing:
    print("missing slots:")
    for (modality, label, kind), why in sorted(mining_missing.items()):
        print(f"  {modality:10} {label:11} {kind:10} {why}")

# Ranking by mean recall over RANK_KINDS, with the step down to the next group in
# percentage points of recall and as a fraction of the higher group.
for modality in ["text", "multimodal"]:
    groups = sorted([label for label, _, _ in MINING_GROUPS],
                    key=lambda g: rank_score(modality, g), reverse=True)
    print(f"\n{modality}: mean Recall@{k_for(modality)} over {' + '.join(RANK_KINDS)}")
    previous = None
    for g in groups:
        score = rank_score(modality, g)
        line = f"  {g:<26}{score:.4f}"
        if previous is not None:
            delta = score - previous
            line += f"   {100 * delta:+.2f} pp vs. above ({100 * delta / previous:+.2f}%)"
        print(line)
        previous = score

# What the miner did to the train split, per mined dataset.
import glob
reports = sorted(glob.glob("dataset/processed/*_mined-*/mining_report.json"))
if reports:
    rows = []
    for path in reports:
        r = json.load(open(path))
        rows.append({
            "dataset": os.path.basename(os.path.dirname(path)),
            "query_kind": r["query_kind"],
            "hard train rows": r["n_hard_train_rows"],
            "mined": r["n_mined"],
            "mined by rule": r["n_mined_by_rule"] / r["n_hard_train_rows"],
            "mined by fallback": r["n_mined_by_fallback"] / r["n_hard_train_rows"],
            "dropped (no survivor)": r["n_dropped_no_survivor"],
            "drop rate": r["n_dropped_no_survivor"] / r["n_hard_train_rows"],
            "top-1 above threshold": r["n_top1_above_threshold"] / r["n_hard_train_rows"],
            "mined == labeled": r["n_mined_equals_labeled"] / max(r["n_mined"], 1),
            "median rank": r["median_mined_rank"],
        })
    display(pd.DataFrame(rows).round(3))
else:
    print("no mined datasets yet: run mine_hard_negs.py (see paper.sh, negs=mined rows)")

# What the labeling pass measured on the mined negatives, per graded dataset.
reports = sorted(glob.glob("dataset/processed/*_mined-*_graded/labeling_report.json"))
if reports:
    rows = []
    for path in reports:
        r = json.load(open(path))
        rows.append({
            "dataset": os.path.basename(os.path.dirname(path)),
            "mined rows": r["n_mined_rows"],
            "labeled": r["n_labeled"],
            "failed": r["n_failed"],
            "distance 0 (false negative)": r["n_distance_zero"] / r["n_labeled"],
            "clipped at max_distance": r["n_clipped_to_max_distance"] / r["n_labeled"],
            "mean distance": r["mean_distance"],
            "mean fraction violated": r["mean_fraction_violated"],
        })
    display(pd.DataFrame(rows).round(3))


## 6. Full table

Every condition, every cutoff. Also written to paper/figs/all_results.csv.

In [ ]:
table = (results.sort_values(["modality", "query_kind", f"recall@{K_MAIN}"],
                           ascending=[True, True, False])
         .reset_index(drop=True))
table.to_csv(os.path.join(FIG_DIR, "all_results.csv"), index=False)
display(table.round(4))
